# A Second Musical Cipher: Schumann's ABEGG Variations

This notebook introduces a second historical cipher as a comparison with the B-A-C-H example in [`bach_cipher.ipynb`](bach_cipher.ipynb).

The key question we'll explore:

> **Does the same letter always produce the same note?**

The answer is no — and the reason tells us something important about music, language, and cultural convention.

---

## Background: Schumann's Op. 1

In 1830, the 20-year-old Robert Schumann published his very first opus: the *Variations on the Name ABEGG* for solo piano. The theme is built from exactly five notes:

**A – B – E – G – G**

These spell out the name of **Meta von Abegg**, a young woman Schumann had met at a ball in Mannheim (though some scholars think she may be fictional — a charming mystery in itself).

### The cipher Schumann used

Unlike Bach, Schumann used the **standard English/French note-naming system**, where every letter maps directly to its note name:

| Letter | Note |
|:---:|:---:|
| A | A |
| B | B♮ (B-natural) |
| C | C |
| D | D |
| E | E |
| F | F |
| G | G |

So **A – B – E – G – G** encodes directly as the notes **A – B♮ – E – G – G**.

---

## The Critical Difference from Bach

In the B-A-C-H cipher, the letter **B** maps to **B♭** (B-flat) — because in the German naming system, 'B' means B-flat and 'H' means B-natural.

In the ABEGG cipher, the letter **B** maps to **B♮** (B-natural) — the English convention.

| Cipher | Letter B maps to |
|:---:|:---:|
| German (Bach) | B♭ (B-flat, MIDI 70) |
| English (Schumann) | B♮ (B-natural, MIDI 71) |

**One letter. Two different notes. Depending entirely on cultural convention.**

This means a decoder who doesn't know which system was used will get the wrong answer — even if they apply the cipher perfectly.

---
## Part 1: Implementing the ABEGG Cipher

In [ ]:
# Note names keyed by pitch class (0-11)
NOTE_NAMES = {
    0: 'C',  1: 'C#', 2: 'D',  3: 'Eb',
    4: 'E',  5: 'F',  6: 'F#', 7: 'G',
    8: 'Ab', 9: 'A',  10: 'Bb', 11: 'B',
}

# -----------------------------------------------------------------------
# The ABEGG cipher: standard English/French note naming
# Only letters A-G are directly encodable (they are note names)
# -----------------------------------------------------------------------
ENGLISH_CIPHER = {
    'A': 69,  # A4
    'B': 71,  # B4  (B-natural — note: NOT B-flat!)
    'C': 60,  # C4
    'D': 62,  # D4
    'E': 64,  # E4
    'F': 65,  # F4
    'G': 67,  # G4
}

# -----------------------------------------------------------------------
# The BACH (German) cipher, for comparison
# -----------------------------------------------------------------------
GERMAN_CIPHER = {
    'A': 69,  # A4
    'B': 70,  # Bb4  (B-flat  — the German 'B')
    'C': 60,  # C4
    'D': 62,  # D4
    'E': 64,  # E4
    'F': 65,  # F4
    'G': 67,  # G4
    'H': 71,  # B4   (B-natural — the German 'H')
}

def encode(text, cipher):
    """Encode a text string using a given cipher dict."""
    result = []
    for ch in text.upper():
        if ch in cipher:
            result.append(cipher[ch])
        elif ch == ' ':
            result.append(None)  # rest
        else:
            raise ValueError(f"'{ch}' not in cipher")
    return result

def note_names_for(midi_notes):
    """Return a list of note name strings for a sequence of MIDI pitches."""
    return [NOTE_NAMES[n % 12] if n is not None else 'REST' for n in midi_notes]

# Encode ABEGG
abegg_notes = encode('ABEGG', ENGLISH_CIPHER)
print("ABEGG encoded (English cipher):")
for letter, note in zip('ABEGG', abegg_notes):
    print(f"  {letter}  →  {NOTE_NAMES[note % 12]}  (MIDI {note})")

---
## Part 2: Side-by-Side Comparison

In [ ]:
# Compare the two cipher systems directly

print("Letter-to-note mapping: German vs English")
print()
print(f"{'Letter':>8}  {'German (Bach)':>15}  {'English (Schumann)':>20}  {'Same?':>7}")
print("-" * 58)

all_letters = sorted(set(GERMAN_CIPHER) | set(ENGLISH_CIPHER))
for letter in all_letters:
    g = GERMAN_CIPHER.get(letter)
    e = ENGLISH_CIPHER.get(letter)
    g_name = NOTE_NAMES[g % 12] if g is not None else '—'
    e_name = NOTE_NAMES[e % 12] if e is not None else '—'
    same = '✓' if g == e else '✗ DIFFERS'
    print(f"  {letter:>6}  {g_name:>15}  {e_name:>20}  {same:>9}")

In [ ]:
# The crucial question: what does B mean?
print("The letter B in each system:")
print()
print(f"  German  'B'  →  Bb  (B-flat,    MIDI {GERMAN_CIPHER['B']})")
print(f"  English 'B'  →  B   (B-natural,  MIDI {ENGLISH_CIPHER['B']})")
print()
print("These are adjacent semitones — as different as C and C#.")
print()
print("If you decode ABEGG using the German cipher by mistake:")
wrong = encode('ABEGG', GERMAN_CIPHER)
print(f"  ABEGG (German cipher) = {note_names_for(wrong)}")
print(f"  ABEGG (English cipher) = {note_names_for(abegg_notes)}")
print()
print("The first note (A) and notes 3-5 (E,G,G) are the same.")
print("Only the B differs — but that one note changes the musical character entirely.")

---
## Part 3: Writing Both Motifs to MIDI

In [ ]:
%pip install midiutil --quiet

In [ ]:
from midiutil import MIDIFile

def write_midi(midi_notes, filename, tempo=72, duration=1.0, volume=90):
    """Write a list of MIDI pitches to a .mid file."""
    midi = MIDIFile(1)
    midi.addTempo(0, 0, tempo)
    time = 0
    for note in midi_notes:
        if note is not None:
            midi.addNote(0, 0, note, time, duration * 0.9, volume)
        time += duration
    with open(filename, 'wb') as f:
        midi.writeFile(f)
    print(f"Written: {filename}")

# BACH motif (German cipher)
bach_notes = encode('BACH', GERMAN_CIPHER)
write_midi(bach_notes, 'bach_motif.mid', tempo=60)

# ABEGG theme (English cipher)
abegg_notes = encode('ABEGG', ENGLISH_CIPHER)
write_midi(abegg_notes, 'abegg_theme.mid', tempo=80)

print()
print("BACH  (German):  ", note_names_for(bach_notes))
print("ABEGG (English): ", note_names_for(abegg_notes))

Open `bach_motif.mid` and `abegg_theme.mid` in GarageBand, MuseScore, or any MIDI player to hear how different they sound — despite both being name ciphers.

---
## Part 4: Searching for ABEGG in a MIDI File

In [ ]:
# Build a sample MIDI that contains both motifs, then search for each

# A short melody: ABEGG theme, a bridge, then BACH motif, then a cadence
Bb4, A4, B4 = 70, 69, 71
C4, D4, E4, F4, G4 = 60, 62, 64, 65, 67
C5, D5 = 72, 74

combined_melody = [
    # ABEGG (notes: A B E G G)
    A4, B4, E4, G4, G4,
    # Bridge
    F4, E4, D4, C4, D4, E4,
    # BACH (notes: Bb A C B)
    Bb4, A4, C4, B4,
    # Closing
    C5, D5, C5, B4, A4, G4,
]

write_midi(combined_melody, 'combined_melody.mid', tempo=80)
print()
print("Full melody:", note_names_for(combined_melody))

In [ ]:
%pip install pretty_midi --quiet

In [ ]:
import pretty_midi

def extract_note_sequence(midi_file):
    """Extract pitch numbers from a MIDI file, sorted by onset time."""
    pm = pretty_midi.PrettyMIDI(midi_file)
    notes = []
    for instrument in pm.instruments:
        if not instrument.is_drum:
            for note in instrument.notes:
                notes.append((note.start, note.pitch))
    notes.sort()
    return [pitch for (_, pitch) in notes]

def search_motif(note_sequence, motif, label='motif'):
    """Search for a motif by pitch class, return positions of matches."""
    motif_pc = [n % 12 for n in motif]
    seq_pc   = [n % 12 for n in note_sequence]
    n = len(motif_pc)
    matches = [i for i in range(len(seq_pc) - n + 1)
               if seq_pc[i:i+n] == motif_pc]
    print(f"  {label:20s}: {note_names_for(motif)}")
    print(f"  {'':20s}  → {len(matches)} match(es) at positions {matches}")
    return matches

# Search the combined melody for both ciphers
seq = extract_note_sequence('combined_melody.mid')
print("Searching combined_melody.mid for both motifs:")
print()
search_motif(seq, encode('ABEGG', ENGLISH_CIPHER), label='ABEGG (English)')
print()
search_motif(seq, encode('BACH',  GERMAN_CIPHER),  label='BACH (German)')

---
## Part 5: What Happens if You Use the Wrong Cipher?

Let's try to find ABEGG using the **German** cipher — the kind of error a naive decoder would make.

In [ ]:
# ABEGG encoded correctly (English cipher)
abegg_english = encode('ABEGG', ENGLISH_CIPHER)  # A Bb E G G  — NO: B=B♮ here

# What ABEGG looks like if encoded with the German cipher by mistake
# (B in German = Bb, so ABEGG in German = A Bb E G G)
abegg_german  = encode('ABEG', GERMAN_CIPHER) + [GERMAN_CIPHER['G']]  
# Note: 'B' in German cipher = Bb = MIDI 70

print("ABEGG under different cipher systems:")
print(f"  English cipher (correct):  {note_names_for(abegg_english)}")
print(f"  German cipher  (wrong):    {note_names_for(abegg_german)}")
print()

print("Now search the melody for ABEGG using the WRONG (German) cipher:")
print()
wrong_motif = encode('ABEGG', GERMAN_CIPHER)
print(f"  Wrong motif to search for: {note_names_for(wrong_motif)}")
matches = search_motif(seq, wrong_motif, label='ABEGG (German — wrong)')
print()
if not matches:
    print("  → Not found! The correct motif is invisible if you use the wrong system.")

This is a key result: **a cipher only works if the decoder knows the convention**.

An automated search that doesn't know whether to use the German or English system must try both — doubling the search space. With many cipher systems in play, the space grows rapidly.

---
## Part 6: Side-by-Side Summary

In [ ]:
# Final comparison table
print("="*62)
print(f"{'Property':<28}  {'BACH':>14}  {'ABEGG':>14}")
print("="*62)

rows = [
    ("Composer",           "J.S. Bach",       "R. Schumann"),
    ("Year",               "c. 1740s",         "1830"),
    ("Work",               "Art of Fugue",     "Op. 1 Variations"),
    ("Cipher system",      "German",           "English/French"),
    ("Letters encoded",    "4 (B,A,C,H)",      "5 (A,B,E,G,G)"),
    ("'B' means",          "Bb (B-flat)",      "B♮ (B-natural)"),
    ("Notes",              "Bb–A–C–B",         "A–B–E–G–G"),
    ("Subject encodes?",   "Composer's name",  "Patron's name"),
]

for label, bach_val, abegg_val in rows:
    print(f"  {label:<26}  {bach_val:>14}  {abegg_val:>14}")

print("="*62)
print()
print("Both are name ciphers. Both hide text in music.")
print("But they use different conventions — and the difference matters.")

---
## Discussion Questions

1. Schumann's dedication may be to a fictional person. Does that change what the cipher *means*? Is there a difference between a cipher that encodes a real name and one that encodes an invented one?

2. The German and English systems agree on every note **except** B. Why might this be? (Hint: look up the history of the German 'B' and 'H' notation.)

3. In Part 5 we saw that using the wrong cipher makes the motif invisible. What does this tell us about automated musical analysis — specifically about tools that assume a single universal note-naming system?

4. Both BACH and ABEGG have been used as homage ciphers by later composers. Liszt, Schumann, and Busoni all quoted BACH. If you were writing a piece today, which cipher system would you use to hide your own name — and why?

---
*Humanities Programming class — University of Oxford*